In [0]:
# Reload the raw "bronze" dataset so this notebook runs independently
df = spark.read.csv(
    "/Volumes/loan_default_workspace/default/raw_data",
    header=True,
    inferSchema=True
)

print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")

Rows: 148670
Columns: 34


In [0]:
# The numeric columns that have nulls (from our Day 1 profiling)
numeric_cols = [
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "dtir1",
    "property_value",
    "LTV",
    "income"
]

# Calculate the median for each column and fill the nulls with it
for column in numeric_cols:
    median_value = df.approxQuantile(column, [0.5], 0.01)[0]
    df = df.fillna({column: median_value})
    print(f"{column}: filled nulls with median = {median_value}")

rate_of_interest: filled nulls with median = 3.99
Interest_rate_spread: filled nulls with median = 0.3832
Upfront_charges: filled nulls with median = 2557.0
dtir1: filled nulls with median = 39.0
property_value: filled nulls with median = 418000.0
LTV: filled nulls with median = 75.10245902
income: filled nulls with median = 5700.0


In [0]:
from pyspark.sql.functions import col

# Loop through every column and count its nulls one at a time
remaining_null_cols = []

for column in df.columns:
    null_count = df.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"{column}: {null_count} nulls")
        remaining_null_cols.append(column)

print(f"\nColumns we still need to clean: {remaining_null_cols}")


Columns we still need to clean: []


In [0]:
from pyspark.sql.functions import when, col

# Create a new column called risk_tier based on Credit_Score bands
df = df.withColumn(
    "risk_tier",
    when(col("Credit_Score") >= 740, "Excellent")
    .when(col("Credit_Score") >= 670, "Good")
    .when(col("Credit_Score") >= 580, "Fair")
    .otherwise("Poor")
)

# Quick check — how many customers fall into each tier?
df.groupBy("risk_tier").count().show()

+---------+-----+
|risk_tier|count|
+---------+-----+
|     Poor|29737|
|     Fair|33548|
|Excellent|59612|
|     Good|25773|
+---------+-----+



In [0]:
# Create a new column called ltv_bucket based on the LTV value
df = df.withColumn(
    "ltv_bucket",
    when(col("LTV") < 60, "Low")
    .when(col("LTV") < 80, "Medium")
    .when(col("LTV") < 100, "High")
    .otherwise("Very High")
)

# Quick check — how many loans in each LTV bucket?
df.groupBy("ltv_bucket").count().show()

+----------+-----+
|ltv_bucket|count|
+----------+-----+
| Very High| 1799|
|      High|49313|
|       Low|32565|
|    Medium|64993|
+----------+-----+



In [0]:
# Create a new column called dti_bucket based on the dtir1 value
df = df.withColumn(
    "dti_bucket",
    when(col("dtir1") < 36, "Low")
    .when(col("dtir1") < 43, "Medium")
    .otherwise("High")
)

# Quick check — how many loans in each DTI bucket?
df.groupBy("dti_bucket").count().show()

+----------+-----+
|dti_bucket|count|
+----------+-----+
|      High|44716|
|       Low|42730|
|    Medium|61224|
+----------+-----+



In [0]:
# Save the cleaned dataframe as a permanent Delta table
spark.sql("CREATE SCHEMA IF NOT EXISTS loan_default_workspace.default")
df.write.mode("overwrite").saveAsTable("loan_default_workspace.default.loans_silver")


print("Silver table saved successfully.")

Silver table saved successfully.


In [0]:
# Reload the silver table from disk to confirm it actually saved
silver_df = spark.table("loan_default_workspace.default.loans_silver")

# Check the row count
row_count = silver_df.count()
print(f"Rows in silver table: {row_count}")

# Check the column count
column_count = len(silver_df.columns)
print(f"Columns in silver table: {column_count}")

# Show all the column names so we can see our 3 new derived columns
print("\nAll columns:")
for column_name in silver_df.columns:
    print(f"  - {column_name}")

Rows in silver table: 148670
Columns in silver table: 37

All columns:
  - ID
  - year
  - loan_limit
  - Gender
  - approv_in_adv
  - loan_type
  - loan_purpose
  - Credit_Worthiness
  - open_credit
  - business_or_commercial
  - loan_amount
  - rate_of_interest
  - Interest_rate_spread
  - Upfront_charges
  - term
  - Neg_ammortization
  - interest_only
  - lump_sum_payment
  - property_value
  - construction_type
  - occupancy_type
  - Secured_by
  - total_units
  - income
  - credit_type
  - Credit_Score
  - co-applicant_credit_type
  - age
  - submission_of_application
  - LTV
  - Region
  - Security_Type
  - Status
  - dtir1
  - risk_tier
  - ltv_bucket
  - dti_bucket
